In [ ]:
%run nlp_utils.py

In [ ]:
!jupyter nbextension enable --py widgetsnbextension

In [ ]:
import tensorflow as tf
from tensorflow import keras

from nlp_utils import utils_plot_keras_training, add_preprocessed_text
from nlp_utils import create_stopwords, add_text_length, plot_distributions

from sklearn.model_selection import train_test_split
import nltk
import pandas as pd

import keras_tuner
from keras_tuner import HyperModel

import keras_nlp

import numpy as np
import math
import random

import os
import pickle

from IPython.display import display, HTML

from collections import Counter
from tqdm.notebook import trange, tqdm
from multiprocessing import Pool

from itertools import repeat
from functools import partial

In [ ]:
train = pd.read_csv('data/train.csv', index_col=0)
validate = pd.read_csv('data/validate.csv', index_col=0)
test = pd.read_csv('data/test.csv', index_col=0)

In [ ]:
# train = pd.read_csv('train.csv', index_col=0)
# validate = pd.read_csv('validate.csv', index_col=0)
# test = pd.read_csv('test.csv', index_col=0)

In [ ]:
tokenizer_train_corpus = pd.concat([train, validate], axis=0, ignore_index=True)

In [ ]:
# train, validate2 = train_test_split(train, test_size=0.1, random_state=42)
# validate = pd.concat([validate, validate2], axis=0, ignore_index=True)

In [ ]:
# validate.info()

In [ ]:
tokenizer_train_corpus.info()

In [ ]:
train_article = train['article_clean'].tolist()
train_highlights = train['highlights_clean'].tolist()

validate_article = validate['article_clean'].tolist()
validate_highlights = validate['highlights_clean'].tolist()

tokenizer_train_article = tokenizer_train_corpus['article_clean'].tolist()
tokenizer_train_highlights = tokenizer_train_corpus['highlights_clean'].tolist()

In [ ]:
def train_sentence_piece(text_samples, vocab_size, model_type, data_type):
    word_piece_ds = tf.data.Dataset.from_tensor_slices(text_samples)

    vocab = keras_nlp.tokenizers.compute_sentence_piece_proto(
        word_piece_ds,
        vocabulary_size=vocab_size,
        model_type=model_type,
        proto_output_file= 'sentencepiece/'+'sentencepiece'  + '_' + data_type + '_' +  'pubmed'  + '.spm',
        lowercase=False
    )

In [ ]:
ART_VOCAB_SIZE = 15_000
HIGH_VOCAB_SIZE = 15_000


MODEL_UNIGRAM = 'unigram'
MODEL_WORD = 'word'

In [ ]:
ART_VOCAB_SIZE = 9_207
HIGH_VOCAB_SIZE = 5_113

In [ ]:
# proto_train_article = train_sentence_piece(
#     tokenizer_train_article,
#     vocab_size=ART_VOCAB_SIZE,
#     model_type=MODEL_UNIGRAM,
#     data_type='train_article'
# )

# proto_train_highlights = train_sentence_piece(
#     tokenizer_train_highlights,
#     vocab_size=HIGH_VOCAB_SIZE,
#     model_type=MODEL_UNIGRAM,
#     data_type='train_highlights'
# )

In [ ]:
# proto_validate_article = train_sentence_piece(
#     validate_article,
#     vocab_size=ENG_VOCAB_SIZE,
#     model_type=MODEL_UNIGRAM,
#     data_type=' validate_article'
# )

# proto_validate_highlights = train_sentence_piece(
#     validate_highlights,
#     vocab_size=ENG_VOCAB_SIZE,
#     model_type=MODEL_UNIGRAM,
#     data_type='validate_highlights'
# )

In [ ]:
train_article_for_tokenizer = tf.data.Dataset.from_tensor_slices([tokenizer_train_article])
train_highlights_for_tokenizer = tf.data.Dataset.from_tensor_slices([tokenizer_train_highlights])

article_tokenizer = keras_nlp.tokenizers.SentencePieceTokenizer(proto='sentencepiece/sentencepiece_train_article_pubmed.spm')
highlights_tokenizer = keras_nlp.tokenizers.SentencePieceTokenizer(proto='sentencepiece/sentencepiece_train_highlights_pubmed.spm')

train_article_sentencepiece = train_article_for_tokenizer.map(article_tokenizer)
train_highlights_sentencepiece = train_highlights_for_tokenizer.map(highlights_tokenizer)

In [ ]:
article_tokenizer.vocabulary_size()

In [ ]:
highlights_tokenizer.vocabulary_size()

In [ ]:
highlights_tokenizer.token_to_id('_')

In [ ]:
train_highlights_sentencepiece

In [ ]:
from string import ascii_lowercase

r = []
for c in ascii_lowercase:
    tok = article_tokenizer.token_to_id(c)
    r.append(tok)
print(len(r))

In [ ]:
len(ascii_lowercase)

In [ ]:
train_article_sentencepiece

In [ ]:
for i in range(20):
    print(article_tokenizer.id_to_token(i))

In [ ]:
list(train_article_sentencepiece)[0][0]

In [ ]:
def train_word_piece(text_samples, vocab_size, reserved_tokens, data_type, article_type):
    word_piece_ds = tf.data.Dataset.from_tensor_slices(text_samples)
    vocab = keras_nlp.tokenizers.compute_word_piece_vocabulary(
        word_piece_ds.batch(1000).prefetch(2),
        vocabulary_size=vocab_size,
        reserved_tokens=reserved_tokens,
        vocabulary_output_file="wordpiece/" + "wordpiece" + '_' + data_type + '_' + article_type + '_' + 'pubmed' + '.spm'

    )
    return vocab

In [ ]:
reserved_tokens = ["[PAD]", "[UNK]", "[START]", "[END]"]

# train_word_piece(tokenizer_train_highlights, HIGH_VOCAB_SIZE, reserved_tokens=reserved_tokens, data_type='train', article_type='highlights')

In [ ]:
with open('wordpiece/wordpiece_train_highlights_pubmed', 'r', encoding='utf-8') as file:
    word_piece_vocab_highlights = [line.strip() for line in file.readlines()]

In [ ]:
word_piece_vocab_highlights[:20]

In [ ]:
# train_word_piece(tokenizer_train_article, ART_VOCAB_SIZE, reserved_tokens=reserved_tokens, data_type='train', article_type='article')

In [ ]:
with open('wordpiece/wordpiece_train_article_pubmed', 'r', encoding='utf-8') as file:
    word_piece_vocab_article = [line.strip() for line in file.readlines()]

In [ ]:
word_piece_atricle_tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=word_piece_vocab_article, lowercase=False
)

word_piece_highlights_tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=word_piece_vocab_highlights, lowercase=False
)


In [ ]:
word_piece_atricle_tokenizer.vocabulary_size()

In [ ]:
word_piece_highlights_tokenizer.vocabulary_size()

In [ ]:
vocab = list(word_piece_atricle_tokenizer.vocabulary)

In [ ]:
for word in vocab:
    if ('#' not in word) and len(word) != 1:
        print(word) 

In [ ]:
sentence_piece_article_tokenizer = keras_nlp.tokenizers.SentencePieceTokenizer(proto='sentencepiece/sentencepiece_train_article_pubmed.spm')
sentence_piece_highlights_tokenizer = keras_nlp.tokenizers.SentencePieceTokenizer(proto='sentencepiece/sentencepiece_train_highlights_pubmed.spm')

In [ ]:
MAX_SEQUENCE_LENGTH = 64

In [ ]:
def preprocess_batch_wordpiece(article, highlights):

    article = word_piece_atricle_tokenizer(article)
    highlights = word_piece_highlights_tokenizer(highlights)

    article_start_end_packer = keras_nlp.layers.StartEndPacker(
        sequence_length=MAX_SEQUENCE_LENGTH,
        pad_value=word_piece_atricle_tokenizer.token_to_id("[PAD]"),
    )
    article = article_start_end_packer(article)

    highlights_start_end_packer = keras_nlp.layers.StartEndPacker(
        sequence_length=MAX_SEQUENCE_LENGTH + 1,
        start_value=word_piece_highlights_tokenizer.token_to_id("[START]"),
        end_value=word_piece_highlights_tokenizer.token_to_id("[END]"),
        pad_value=word_piece_highlights_tokenizer.token_to_id("[PAD]"),
    )
    highlights = highlights_start_end_packer(highlights)

    return (
        {
            "encoder_inputs": article,
            "decoder_inputs": highlights[:, :-1],
        },
        highlights[:, 1:],
    )

def preprocess_batch_sentencepiece(article, highlights):
    print(article)

    article = sentence_piece_article_tokenizer(article)
    highlights = sentence_piece_highlights_tokenizer(highlights)

    article_start_end_packer = keras_nlp.layers.StartEndPacker(
        sequence_length=MAX_SEQUENCE_LENGTH,
        pad_value=tf.cast(tf.numpy_function(sentence_piece_article_tokenizer.token_to_id, [tf.constant("<pad>")], tf.int64), dtype=tf.int32),
    )
    article = article_start_end_packer(article)

    print(highlights)

    highlights_start_end_packer = keras_nlp.layers.StartEndPacker(
        sequence_length=MAX_SEQUENCE_LENGTH + 1,
        start_value = tf.cast(tf.numpy_function(sentence_piece_highlights_tokenizer.token_to_id, [tf.constant("<s>")], tf.int64), tf.int32),
        end_value = tf.cast(tf.numpy_function(sentence_piece_highlights_tokenizer.token_to_id, [tf.constant("</s>")], tf.int64), tf.int32),
        pad_value = tf.cast(tf.numpy_function(sentence_piece_highlights_tokenizer.token_to_id, [tf.constant("<pad>")], tf.int64), tf.int32),
    )
    highlights = highlights_start_end_packer(highlights)
    

    return (
        {
            "encoder_inputs": article,
            "decoder_inputs": highlights[:, :-1],
        },
        highlights[:, 1:],
    )

def data_generator(articles, highlights):
    for article, highlight in zip(articles, highlights):
        yield article, highlight

def make_dataset(pairs, preprocess_batch, batch_size=32):
    articles, highlights = pairs
    dataset = tf.data.Dataset.from_tensor_slices((articles, highlights))
    
    dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(articles, highlights),
    output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.string),  # article
        tf.TensorSpec(shape=(), dtype=tf.string)   # highlight
    )
    )
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(preprocess_batch, num_parallel_calls=tf.data.AUTOTUNE)
    
    # dataset = dataset.map(preprocess_batch)
    
    res = dataset.shuffle(2048).prefetch(16).cache()

    return res 

In [ ]:
batch_size = 32
validate_pairs = validate_article, validate_highlights
train_pairs = train_article, train_highlights

In [ ]:
batch_size = 32
sequence_length = 64
dropout_rate = 0.1
num_heads = 6
intermediate_dim = 64
embed_dim = 64
number_of_layers = 3

In [ ]:
class MyHyperModel(HyperModel):
    
    def build(self, hp):
        sequence_length = 64
        dropout_rate = 0.1
        num_heads = 6
        intermediate_dim = 64
        embed_dim = 64
        number_of_layers = 3
        
        encoder_inputs = keras.Input(
        shape=(None,),
        dtype="int64", 
        name="encoder_inputs")

        x = keras_nlp.layers.TokenAndPositionEmbedding(
            vocabulary_size=ART_VOCAB_SIZE,
            sequence_length=sequence_length,
            embedding_dim=embed_dim,
            mask_zero=True,
        )(encoder_inputs)
        

        for _ in range(number_of_layers):
            encoder_outputs = keras_nlp.layers.TransformerEncoder(
                intermediate_dim=intermediate_dim, 
                num_heads=num_heads,
                )(inputs=x)

        x = keras.layers.BatchNormalization()(x)
        encoder = keras.Model(encoder_inputs, encoder_outputs)

        
        # Decoder
        decoder_inputs = keras.Input(
            shape=(None,),
            dtype="int64",
            name="decoder_inputs"
        )

        encoded_seq_inputs = keras.Input(
            shape=(None, embed_dim),
            name="decoder_state_inputs"
        )

 
        x = keras_nlp.layers.TokenAndPositionEmbedding(
            vocabulary_size=HIGH_VOCAB_SIZE,
            sequence_length=sequence_length,
            embedding_dim=embed_dim,
            mask_zero=True,

        )(decoder_inputs)
        
        
        for _ in range(number_of_layers):
            x = keras_nlp.layers.TransformerDecoder(
                intermediate_dim=intermediate_dim,
                num_heads=num_heads,
                # dropout=dropout_rate
            )(decoder_sequence=x, encoder_sequence=encoded_seq_inputs)


        x = keras.layers.Dropout(dropout_rate)(x)
        x = keras.layers.BatchNormalization()(x) 

        
        decoder_outputs = keras.layers.Dense(
            HIGH_VOCAB_SIZE,
            activation="softmax",
            )(x)

        decoder = keras.Model(
            [
                decoder_inputs,
                encoded_seq_inputs,
            ],
            decoder_outputs,
        )

        decoder_outputs = decoder([decoder_inputs, encoder_outputs])

        transformer = keras.Model(
            [encoder_inputs, decoder_inputs],
            decoder_outputs,
            name="transformer",
        )


        optimizer = tf.keras.optimizers.Adafactor(
            learning_rate=0.01,
            beta_2_decay=-1,
            epsilon_1=1e-30,
            epsilon_2=0.001,
            clip_threshold=1.0,
            relative_step=True,
            weight_decay=None,
            # clipnorm=1,
            clipvalue=1,
            global_clipnorm=1,
            use_ema=False,
            ema_momentum=0.99,
            ema_overwrite_frequency=None,
            jit_compile=True,
            name="Adafactor",
        )

        transformer.compile(
            loss="sparse_categorical_crossentropy", 
            optimizer=optimizer,
            metrics=[keras_nlp.metrics.Perplexity(name="perplexity")]
        )

        return transformer
    


    def fit(self, hp, model, train_pairs,  validation_pairs, callbacks=None, **kwargs):
        batch_size = 32
        train_dataset = make_dataset(train_pairs, preprocess_batch_wordpiece, batch_size)

        val_dataset = make_dataset(validation_pairs, preprocess_batch_wordpiece, batch_size)
        
        my_callback = keras.callbacks.EarlyStopping(monitor='val_perplexity', patience=4, mode='auto', min_delta=1)
        with tf.device('/GPU:0'):
            res = model.fit(
                train_dataset,
                shuffle=True,
                validation_data=val_dataset,
                callbacks=[my_callback],
                **kwargs,
            )


        return res

In [ ]:
# tuner = keras_tuner.Hyperband(
#     executions_per_trial=1,
#     max_epochs=30,
#     factor=3,
#     hyperband_iterations=3,
#     hypermodel=MyHyperModel(),
#     objective=keras_tuner.Objective("val_perplexity", direction="min"),
#     directory="results3",
#     project_name="custom_training",
#     max_retries_per_trial=0,
#     max_consecutive_failed_trials=3,
#     overwrite=False,
#     allow_new_entries=True
# )

In [ ]:
tuner = keras_tuner.BayesianOptimization(
    hypermodel=MyHyperModel(),
    objective=keras_tuner.Objective("val_perplexity", direction="min"),
    max_trials=40,
    overwrite=False,
    alpha=0.0001,
    beta=2.6,
    seed=None,
    tune_new_entries=True,
    allow_new_entries=True,
    max_retries_per_trial=0,
    max_consecutive_failed_trials=1,
    directory="test_Adafactor_2",
    project_name="custom_training",
)

In [ ]:
validate_pairs = validate_article, validate_highlights
train_pairs = train_article, train_highlights

In [ ]:
# my_callback = keras.callbacks.EarlyStopping(monitor='val_perplexity', patience=6, mode='auto', min_delta=0, restore_best_weights=True)
# with tf.device('/GPU:0'):
#     tuner.search(train_pairs=train_pairs, validation_pairs=validate_pairs, callbacks=[my_callback], epochs=30)

In [ ]:
best_hps=tuner.get_best_hyperparameters()[0]

In [ ]:
validate_pairs = validate_article, validate_highlights
train_pairs = train_article, train_highlights

In [ ]:
model = tuner.hypermodel.build(best_hps)

In [ ]:
batch_size=32

In [ ]:
train_dataset = make_dataset(train_pairs, preprocess_batch_wordpiece, batch_size)

val_dataset = make_dataset(validate_pairs, preprocess_batch_wordpiece, batch_size)

In [ ]:
train_article = train['article_clean'].tolist()
train_highlights = train['highlights_clean'].tolist()

validate_article = validate['article_clean'].tolist()
validate_highlights = validate['highlights_clean'].tolist()

In [ ]:
model = keras.models.load_model("transformer_wordpiece_model.h5")

In [ ]:
# my_callback = keras.callbacks.EarlyStopping(monitor='val_perplexity', patience=10, mode='auto', min_delta=0, restore_best_weights=True)
# callbacks=[my_callback]
#     # model.summary()

# with tf.device('/GPU:0'):
#     model.fit(
#         x=train_dataset,
#         validation_data=val_dataset,
#         callbacks=my_callback,
#         epochs=100,
#         shuffle=True
#     )

In [ ]:
# model.save("transformer_wordpiece_model.h5")

In [ ]:
# model = keras.models.load_model("transformer_wordpiece_model.h5")


In [ ]:
# with open('history_wordpiece.pkl', 'wb') as file:
#     pickle.dump(model.history, file)

In [ ]:
with open('history_wordpiece.pkl', 'rb') as file:
    saved_history = pickle.load(file)

In [ ]:
utils_plot_keras_training(saved_history)


In [ ]:
from nlp_utils import create_stopwords, add_preprocessed_text

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

In [ ]:
test = pd.read_csv('data/test.csv', index_col=0)

In [ ]:
lst_stopwords = create_stopwords()
lst_stopwords = lst_stopwords + ["say","said","new","wa","ha", 'p', 'ci', 'v', 'n', 'itti', 'vs', 'pc', 'auc', 'itti']

columns = ['article', 'highlights']

params = {
    'punkt': True,
    'lower': True,
    'slang': True,
    'lst_stopwords': lst_stopwords,
    'lst_regex': [r"(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|^rt|http.+?", r"\d+"],
    'lemm': True,
    # 'stemm' : True
}

for column in columns:
    params['column'] = column
    test = add_preprocessed_text(test, **params).reset_index(drop=True)

In [ ]:

test_articles = test['article_clean'].tolist()
test_highlights= test['highlights_clean'].tolist()

In [ ]:
test_pairs = list(zip(test_articles, test_highlights))

In [ ]:
# batch_size = best_hps.get_config()['values']['batch_size']
batch_size = 32

In [ ]:
NUM_BEAMS = 5

def decode_sequences(input_sentences, model):
    batch_size = tf.shape(input_sentences)[0]
    # Tokenize the encoder input.
    encoder_input_tokens = word_piece_atricle_tokenizer(input_sentences).to_tensor(
        shape=(None, MAX_SEQUENCE_LENGTH)
    )
    encoder_input_tokens = tf.repeat(encoder_input_tokens, NUM_BEAMS, axis=0)
    # Define a function that outputs the next token's probability given the
    # input sequence.
    def next(prompt, cache, index):
        logits = model([encoder_input_tokens, prompt])[:, index - 1, :]
        # Ignore hidden states for now; only needed for contrastive search.
        hidden_states = None
        return logits, hidden_states, cache

    # Build a prompt of length 40 with a start token and padding tokens.
    length = 30
    start = tf.fill((batch_size, 1), word_piece_highlights_tokenizer.token_to_id("[START]"))
    pad = tf.fill((batch_size, length - 1), word_piece_highlights_tokenizer.token_to_id("[PAD]"))
    prompt = tf.concat((start, pad), axis=-1)

    sampler = keras_nlp.samplers.BeamSampler(num_beams=5)
    
    output_tokens = sampler(
        next,
        prompt,
        end_token_id=word_piece_highlights_tokenizer.token_to_id("[END]"),
        index=1,
    )

    generated_sentences = word_piece_highlights_tokenizer.detokenize(output_tokens)
    
    return generated_sentences

In [ ]:
test_article_texts = [article for article in test['article_clean'].tolist()]
test_highlight_texts = [article for article in test['highlights_clean'].tolist()]


for i in range(3):
    input_sentence = random.choice(test_article_texts)
    translated = decode_sequences(tf.constant([input_sentence]), model)
    # print(translated[0])
    translated = translated.numpy()[0].decode("utf-8")
    translated = (
        translated.replace("[PAD]", "")
        .replace("[START]", "")
        .replace("[END]", "")
        .strip()
    )
    print(f"** Example {i} **")
    print(input_sentence)
    print(translated)

In [ ]:
from nlp_utils import display_string_matching

In [ ]:
rouge_1_metric = keras_nlp.metrics.RougeN(order=1,use_stemmer=True)
# rouge_2 = keras_nlp.metrics.RougeN(order=2,use_stemmer=True)
rouge_l_metric = keras_nlp.metrics.RougeL(use_stemmer=True, name="rouge-l")


roug_l = keras_nlp.metrics.RougeL()

# test_pair = random.choice(test_pairs)

max_res = 0 

columns = ['Rouge-l', 'Rouge-1']
WordPiece = pd.DataFrame(columns=columns)

for _ in range(10):

    n = 10
    picked_elements = random.sample(test_pairs, n)

    for test_pair in picked_elements:
        input_sentence = test_pair[0]
        reference_sentence = test_pair[1]

        translated_sentence = decode_sequences(tf.constant([input_sentence]), model)
        translated_sentence = translated_sentence.numpy()[0].decode("utf-8")
        translated_sentence = (
            translated_sentence.replace("[PAD]", "")
            .replace("[START]", "")
            .replace("[END]", "")
            .strip()
        )

        rouge_l = rouge_l_metric(reference_sentence, translated_sentence)
        rouge_1 = rouge_1_metric(reference_sentence, translated_sentence)



    # Append the row to the DataFrame
    test =  rouge_1['f1_score'].numpy()
    print(f'test: {test}')
    print(f'max res: {max_res}')
    print('\n')
    
    rouge_1_res =  rouge_1['f1_score'].numpy()
    roug_l_res  =  rouge_l['f1_score'].numpy()
    new_row = [roug_l_res , rouge_1_res]
    WordPiece.loc[len(WordPiece)] = new_row
    if max_res < test:
        if max_res == 0:
            min_res = max_res
            rouge_1_min_res =  rouge_1['f1_score']
            roug_l_min_res =  rouge_l['f1_score']
        max_res = test

        # Create a row of numbers

# print("ROUGE-1 Score: ",rouge_1_res)
# print("ROUGE-2 Score: ", rouge_2.result()['f1_score'])
# print("ROUGE-L Score: ", roug_l_res)

In [ ]:
WordPiece = WordPiece.sort_values('Rouge-l', ascending=False).reset_index(drop=True)

In [ ]:
WordPiece



In [ ]:
value = []

for experiment in metrics:
    for index, pair in enumerate(experiment):
        if index == 0:
            pair['f1_score']

In [ ]:
train_dataset = make_dataset(train_pairs, preprocess_batch_sentencepiece, batch_size)

val_dataset = make_dataset(validate_pairs, preprocess_batch_sentencepiece, batch_size)


In [ ]:
best_hps=tuner.get_best_hyperparameters()[0]

In [ ]:
# model_2 = tuner.hypermodel.build(best_hps)

In [ ]:
model_2 = keras.models.load_model('transformer_sentencepiece_model.h5')

In [ ]:
# my_callback = keras.callbacks.EarlyStopping(monitor='val_perplexity', patience=4, mode='auto', min_delta=0, restore_best_weights=True)

# with tf.device('/GPU:0'):
#     model_2.fit(
#         x=train_dataset,
#         validation_data=val_dataset,
#         callbacks=my_callback,
#         epochs=150,
#         shuffle=True
#     )

# model.save("transformer_sentencepiece_model.h5")

In [ ]:
# model_2.save("transformer_sentencepiece_model.h5")

In [ ]:
# with open('history_sentencepiece.pkl', 'wb') as file:
#     pickle.dump(model_2.history, file)

In [ ]:
with open('history_sentencepiece.pkl', 'rb') as file:
    saved_history = pickle.load(file)

In [ ]:
utils_plot_keras_training(saved_history)

In [ ]:
# test_article_texts = [article for article in test['article_clean'].tolist()]
# test_highlight_texts = [article for article in test['highlights_clean'].tolist()]


for i in range(3):
    input_sentence = random.choice(test_article_texts)
    translated = decode_sequences(tf.constant([input_sentence]), model_2)
    # print(translated[0])
    translated = translated.numpy()[0].decode("utf-8")
    translated = (
        translated.replace("[PAD]", "")
        .replace("[START]", "")
        .replace("[END]", "")
        .strip()
    )
    print(f"** Example {i} **")
    print(input_sentence)
    print(translated)

In [ ]:
rouge_1_metric = keras_nlp.metrics.RougeN(order=1,use_stemmer=True)
# rouge_2 = keras_nlp.metrics.RougeN(order=2,use_stemmer=True)
rouge_l_metric = keras_nlp.metrics.RougeL(use_stemmer=True, name="rouge-l")


roug_l = keras_nlp.metrics.RougeL()

# test_pair = random.choice(test_pairs)

max_res = 0 

columns = ['Rouge-l', 'Rouge-1']
SentencePiece = pd.DataFrame(columns=columns)

for _ in range(10):

    n = 10
    picked_elements = random.sample(test_pairs, n)

    for test_pair in picked_elements:
        input_sentence = test_pair[0]
        reference_sentence = test_pair[1]

        translated_sentence = decode_sequences(tf.constant([input_sentence]), model_2)
        translated_sentence = translated_sentence.numpy()[0].decode("utf-8")
        translated_sentence = (
            translated_sentence.replace("[PAD]", "")
            .replace("[START]", "")
            .replace("[END]", "")
            .strip()
        )

        rouge_l = rouge_l_metric(reference_sentence, translated_sentence)
        rouge_1 = rouge_1_metric(reference_sentence, translated_sentence)



    # Append the row to the DataFrame
    test =  rouge_1['f1_score'].numpy()
    print(f'test: {test}')
    print(f'max res: {max_res}')
    print('\n')
    
    rouge_1_res =  rouge_1['f1_score'].numpy()
    roug_l_res  =  rouge_l['f1_score'].numpy()
    new_row = [roug_l_res , rouge_1_res]
    SentencePiece.loc[len(SentencePiece)] = new_row
    if max_res < test:
        if max_res == 0:
            min_res = max_res
            rouge_1_min_res =  rouge_1['f1_score']
            roug_l_min_res =  rouge_l['f1_score']
        max_res = test

# print("ROUGE-1 Score: ",rouge_1_res)
# print("ROUGE-2 Score: ", rouge_2.result()['f1_score'])
# print("ROUGE-L Score: ", roug_l_res)

In [ ]:
SentencePiece = SentencePiece.sort_values('Rouge-l', ascending=False).reset_index(drop=True)
SentencePiece

In [ ]:
increase_percentage = abs((WordPiece - SentencePiece) / SentencePiece) * 100


In [ ]:
increase_percentage.mean()